In [1]:
import gc
import sklearn
import numpy as np
import keras_tuner
import tensorflow as tf
import matplotlib.pyplot as plt

import os, sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("Modules"))))

import Modules.constants as constants 
import Modules.ds_loader as ds_loader

train_loader, val_loader, test_loader = ds_loader.load_tf_data()

2025-04-15 10:37:44.052393: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-15 10:37:44.061902: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744706264.072791   37332 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744706264.076451   37332 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1744706264.086207   37332 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

/home/capitan/Documents/Notes/Materials/3rd Year/S2/MEDDEVICESLAB/LAB1/Data/Dataset
[DEBUG] DATA_PATH = /home/capitan/Documents/Notes/Materials/3rd Year/S2/MEDDEVICESLAB/LAB1/Data/Dataset
[ OK ] Loaded 7409 samples with shape (500, 12)
[ OK ] Loaded 1591 samples with shape (500, 12)
[ OK ] Loaded 1588 samples with shape (500, 12)
Unique classes in y: [0 1 2 3]
Datatype: float32 int32
Min and Max of X_train: 0.0, 1.0000001192092896
Min and Max of X_val: -8.730203628540039, 11.054253578186035
Min and Max of X_test: -11.602144241333008, 13.575825691223145
NaNs in X: 0
Infs in X: 0
Class distribution before SMOTE: Counter({np.int32(2): 2721, np.int32(1): 1583, np.int32(3): 1554, np.int32(0): 1551})


 Applying oversampling via SMOTE
Class distribution after SMOTE: Counter({np.int32(0): 2721, np.int32(1): 2721, np.int32(2): 2721, np.int32(3): 2721})


I0000 00:00:1744706273.247776   37332 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2659 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [ ]:
"""print("Unique classes in y:", np.unique(y_train))
print("Datatype:",(X_train.dtype),(y_train.dtype))
print(f"Min and Max of X_train: {np.min(X_train)}, {np.max(X_train)}")
print(f"Min and Max of X_val: {np.min(X_val)}, {np.max(X_val)}")
print(f"Min and Max of X_test: {np.min(X_test)}, {np.max(X_test)}")
print(f"NaNs in X: {np.isnan(X_train).sum()}")
print(f"Infs in X: {np.isinf(X_train).sum()}")
print(f"Class distribution: {Counter(y_train)}")"""

In [ ]:
"""X_train_flat = X_train.reshape((X_train.shape[0],-1))
smote = imblearn.over_sampling.SMOTE(random_state=42)
X_resampled, y_train = smote.fit_resample(X_train_flat, y_train)
X_train = X_resampled.reshape((-1, *X_train.shape[1:]))

print(f"Class distribution: {Counter(y_train)}")"""

In [ ]:
"""# 1-D convolutional ResNet model 
# https://pmc.ncbi.nlm.nih.gov/articles/PMC10128986/#sec012
class Resnet(keras_tuner.HyperModel):
    def residual_block(self, inputs, c_units, p_units, k_units):
        # C1 BLOCK
        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=k_units, strides=1, padding='same')(inputs)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        
        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=k_units, strides=1, padding='same')(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        # SC
        s = tf.keras.layers.Conv1D(filters=c_units, kernel_size=1, strides=1, padding='same')(inputs)
        x = tf.keras.layers.Add()([x, s])
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling1D(p_units, strides=2)(x)
        return x


    def build(self, hp):
        gc.collect()
        tf.keras.backend.clear_session()
        # HYPERPARAMS
        n_layer = 3
        k_units = 3
        p_units = 5

        c_units = hp.Choice("c_units", [64])
        d_units_0 = hp.Choice("d_units_0", [1024])
        d_units_1 = hp.Choice('d_units_coef', [2,4,8])
        dropout_0 = hp.Float('dropout_0', min_value = 0.3, max_value=0.5, step=0.05)
        dropout_1 = hp.Float('dropout_1', min_value = 0.3, max_value=0.5, step=0.05)
        
        # INPUT LAYER
        inputs = tf.keras.Input(shape=(500,12))
        
        # RESIDUALS
        x = self.residual_block(inputs, c_units, p_units, k_units)
        filter_size = c_units
        for i in range(1, n_layer):
            filter_size *= 2  
            x = self.residual_block(x, filter_size, p_units, k_units)

        # CLASSIFIER
        x = tf.keras.layers.Flatten()(x)
        x = tf.keras.layers.Dense(d_units_0, activation='relu')(x)
        x = tf.keras.layers.Dropout(dropout_0)(x)  
        x = tf.keras.layers.Dense(d_units_0 // d_units_1, activation='relu')(x)
        x = tf.keras.layers.Dropout(dropout_1)(x)  

        # OUTPUT
        outputs = tf.keras.layers.Dense(4, activation='softmax')(x)
        
        model = tf.keras.Model(inputs, outputs)
        optimizer = tf.keras.optimizers.Adam(
                        learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-3),
                        weight_decay=hp.Choice('weight_decay',[1e-3,1e-4,1e-5,0.0])
                    )
        model.compile(
            optimizer=optimizer,
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )
        
        
        return model
    
    def fit(self, hp, model, *args, **kwargs):
        return model.fit(
            batch_size= hp.Choice("batch_size", [32]),
            *args,
            **kwargs,
        ) """

In [2]:
class Resnet(keras_tuner.HyperModel):
    def residual_block(self, inputs, c_units, p_units, k_units):
        # C1 BLOCK
        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=6, strides=1, padding='same')(inputs)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling1D(6, 6, padding="same")(x)
        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=k_units, strides=1, padding='same')(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling1D(p_units, p_units, padding="same")(x)
        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=k_units, strides=1, padding='same')(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling1D(p_units, p_units, padding="same")(x)
    
        return x


    def build(self, hp):
        gc.collect()
        tf.keras.backend.clear_session()
        # HYPERPARAMS
        n_layer = 1
        k_units = 3
        p_units = 3

        c_units = hp.Choice("c_units", [32,64,128])
        d_units_0 = hp.Choice("d_units_0", [64,128,256,512])
        d_units_1 = hp.Choice('d_units_coef', [2])
        dropout_0 = hp.Float('dropout_0', min_value = 0.3, max_value=0.5, step=0.05)
        dropout_1 = hp.Float('dropout_1', min_value = 0.3, max_value=0.5, step=0.05)
        
        # INPUT LAYER
        inputs = tf.keras.Input(shape=(5000//constants.WINDOW_SIZE,12))
        
        # RESIDUALS
        x = self.residual_block(inputs, c_units, p_units, k_units)
        filter_size = c_units
        for i in range(1, n_layer):
            #filter_size *= 2  
            x = self.residual_block(x, filter_size, p_units, k_units)

        # CLASSIFIER
        x = tf.keras.layers.Flatten()(x)
        x = tf.keras.layers.Dense(d_units_0, activation='relu')(x)
        x = tf.keras.layers.Dropout(dropout_0)(x)  
        x = tf.keras.layers.Dense(d_units_0 // d_units_1, activation='relu')(x)
        x = tf.keras.layers.Dropout(dropout_1)(x)  

        # OUTPUT
        outputs = tf.keras.layers.Dense(4, activation='softmax')(x)
        
        model = tf.keras.Model(inputs, outputs)
        optimizer = tf.keras.optimizers.Adam(
                        learning_rate=hp.Float('learning_rate', min_value=1e-5, max_value=1e-3),
                        weight_decay=hp.Choice('weight_decay',[1e-3,1e-4,1e-5,0.0])
                    )
        model.compile(
            optimizer=optimizer,
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )
        
        
        return model
    
    def fit(self, hp, model, *args, **kwargs):
        return model.fit(
            batch_size= hp.Choice("batch_size", [32]),
            *args,
            **kwargs,
        ) 

In [3]:
RDIR="src/Results/RES_500_00/" 
MDIR= RDIR + "RES_W500_00.keras"
CDIR= RDIR + "C_RES_W500_00.keras"
CVDIR = RDIR + "RES_W500_00_CV.keras"
tuner = keras_tuner.BayesianOptimization(
    Resnet(),
    max_trials=100,
    overwrite=False,
    objective='val_accuracy',
    directory=RDIR,
    project_name="RES_W500_00",
    )

tuner.search_space_summary()

Search space summary
Default search space size: 7
c_units (Choice)
{'default': 32, 'conditions': [], 'values': [32, 64, 128], 'ordered': True}
d_units_0 (Choice)
{'default': 64, 'conditions': [], 'values': [64, 128, 256, 512], 'ordered': True}
d_units_coef (Choice)
{'default': 2, 'conditions': [], 'values': [2], 'ordered': True}
dropout_0 (Float)
{'default': 0.3, 'conditions': [], 'min_value': 0.3, 'max_value': 0.5, 'step': 0.05, 'sampling': 'linear'}
dropout_1 (Float)
{'default': 0.3, 'conditions': [], 'min_value': 0.3, 'max_value': 0.5, 'step': 0.05, 'sampling': 'linear'}
learning_rate (Float)
{'default': 1e-05, 'conditions': [], 'min_value': 1e-05, 'max_value': 0.001, 'step': None, 'sampling': 'linear'}
weight_decay (Choice)
{'default': 0.001, 'conditions': [], 'values': [0.001, 0.0001, 1e-05, 0.0], 'ordered': True}


In [ ]:
callback_list = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy",mode="max", restore_best_weights=True,patience=5, verbose=0),
    tf.keras.callbacks.ModelCheckpoint(filepath=CDIR,monitor='val_accuracy', save_best_only=True, save_weights_only=False,    
    verbose=0)
]
tuner.search(
    train_loader, 
    epochs = 150,
    validation_data=(val_loader),
    callbacks=callback_list 
)

Trial 8 Complete [00h 00m 13s]
val_accuracy: 0.27141058444976807

Best val_accuracy So Far: 0.50314861536026
Total elapsed time: 00h 02m 31s

Search: Running Trial #9

Value             |Best Value So Far |Hyperparameter
64                |64                |c_units
128               |64                |d_units_0
2                 |2                 |d_units_coef
0.35              |0.3               |dropout_0
0.35              |0.45              |dropout_1
0.00052913        |3.9517e-05        |learning_rate
0.0001            |0                 |weight_decay
32                |32                |batch_size

Epoch 1/150
341/341 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.7380 - loss: 1.2853 - val_accuracy: 0.2173 - val_loss: 2.0033
Epoch 2/150
341/341 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5137 - loss: 1.4783 - val_accuracy: 0.2103 - val_loss: 2.1502
Epoch 3/150
341/341 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4256 - loss: 1.4360 - val_accuracy: 0.2305 - val_loss: 1.53

In [ ]:
tuner.results_summary()

In [ ]:
models = tuner.get_best_models(num_models=1)
best_model = models[0]
best_model.summary()
best_model.save(MDIR) 

In [ ]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0] 
print(best_hps.values)

In [ ]:
test_loss, test_accuracy = best_model.evaluate(test_loader, batch_size=32)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

In [ ]:
X_test_list, y_test_list = [], []
X_train_list, y_train_list= [], []

for batch_x, batch_y in test_loader:
    X_test_list.append(batch_x.numpy())
    y_test_list.append(batch_y.numpy())
for batch_x, batch_y in train_loader:
    X_train_list.append(batch_x.numpy())
    y_train_list.append(batch_y.numpy())

X_test = np.concatenate(X_test_list, axis=0)
y_test = np.concatenate(y_test_list, axis=0)
X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

In [ ]:
y_pred = best_model.predict(X_test)

if y_pred.shape[1] == 1:  
    y_pred_binary = (y_pred > 0.5).astype(int)
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred)  
else:
    y_pred_binary = np.argmax(y_pred, axis=1)  
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred, multi_class='ovr')

print("Classification Report (Test Data):")
print(sklearn.metrics.classification_report(y_test, y_pred_binary))
print(f"AUC: {auc}")

y_train_pred = best_model.predict(X_train)
y_train_pred = np.argmax(y_train_pred, axis=1)

print("Classification Report (Train Data):")
print(sklearn.metrics.classification_report(y_train, y_train_pred))

In [ ]:
import seaborn as sns
y_pred_class = np.argmax(y_pred, axis=1)  
cm = sklearn.metrics.confusion_matrix(y_test, y_pred_class, normalize='true')

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
    max_trials=100,
    overwrite=True,
    objective='val_accuracy',
    directory=RDIR,
    project_name="RES_W1000_L33",
    )

tuner.search_space_
plt.title('Confusion Matrix')
plt.show()

In [ ]:
kfold = sklearn.model_selection.KFold(n_splits=10, shuffle=True, random_state=42)
fold_accuracies = []
fold_histories = []

best_accuracy = 0.0
best_model = None  

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train, y_train)):
    print(f"\n--- Fold {fold+1} ---")

    fold_callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)
    ]
    X_tr, X_val_fold = X_train[train_idx], X_train[val_idx]
    y_tr, y_val_fold = y_train[train_idx], y_train[val_idx]

    model = Resnet().build(best_hps)

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val_fold, y_val_fold),
        epochs=100,
        callbacks=fold_callbacks,
        verbose=1
    )

    val_loss, val_accuracy = model.evaluate(X_val_fold, y_val_fold, verbose=0)
    print(f"Fold {fold+1} Validation Accuracy: {val_accuracy:.4f}")
    fold_accuracies.append(val_accuracy)
    fold_histories.append(history)

    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_model = model
        model.save(CVDIR) 
        print(f"Saved best model from Fold {fold+1} with Accuracy: {val_accuracy:.4f}")


In [ ]:
print("Cross-validation accuracies:", fold_accuracies)
print("Average CV accuracy:", np.mean(fold_accuracies))
print("Max CV accuracy:", np.max(fold_accuracies))

In [ ]:
cv_model = tf.keras.models.load_model(CVDIR)
cv_model.evaluate(X_test, y_test)

In [ ]:
test_loss, test_accuracy = cv_model.evaluate(X_test, y_test, batch_size=32)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

In [ ]:
y_pred = cv_model.predict(X_test)

if y_pred.shape[1] == 1:  
    y_pred_binary = (y_pred > 0.5).astype(int)
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred)  
else:
    y_pred_binary = np.argmax(y_pred, axis=1)  
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred, multi_class='ovr')

print("Classification Report (Test Data):")
print(sklearn.metrics.classification_report(y_test, y_pred_binary))
print(f"AUC: {auc}")

y_train_pred = cv_model.predict(X_train)
y_train_pred = np.argmax(y_train_pred, axis=1)

print("Classification Report (Train Data):")
print(sklearn.metrics.classification_report(y_train, y_train_pred))

In [ ]:
y_pred_probs = cv_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("Classification Report (Test Data):")
print(sklearn.metrics.classification_report(y_test, y_pred))

auc = sklearn.metrics.roc_auc_score(y_test, y_pred_probs, multi_class='ovr')
print(f"AUC (Test): {auc:.4f}")

y_train_probs = cv_model.predict(X_train)
y_train_pred = np.argmax(y_train_probs, axis=1)

print("Classification Report (Train Data):")
print(sklearn.metrics.classification_report(y_train, y_train_pred))

In [ ]:
import seaborn as sns

cm = sklearn.metrics.confusion_matrix(y_test, y_pred, normalize='true')

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

In [ ]:

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

ax[0].plot(history.history['accuracy'], label='accuracy')
ax[0].plot(history.history['val_accuracy'], label='val_accuracy')
ax[0].set_title('Accuracy vs Val Accuracy')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Accuracy')
ax[0].legend()

ax[1].plot(history.history['loss'], label='loss')
ax[1].plot(history.history['val_loss'], label='val_loss')
ax[1].set_title('Loss vs Val Loss')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Loss')
ax[1].legend()

plt.tight_layout()
plt.show()